# Bug matcher: QLoRA fine-tune
Qwen3 + Unsloth. Production uses **Qwen3-8B** (one 24 GB GPU, 8k context); this notebook defaults to **Qwen3-0.6B** so it fits a free Colab T4. Input: root-cause analysis + ranker top-5 candidates with evidence probe and graph facts. Output: JSON pick or NONE.

> All data in this notebook is **synthetic** (generated below). The real system runs on private validation data; the code paths are the same, the numbers in the README are from the real data.

In [ ]:
!pip -q install unsloth trl datasets scikit-learn

In [ ]:
%%writefile synthetic.py
"""Synthetic failure generator for the demo notebooks (no real data is published).
Each bug has a signature (2-3 log lines), a domain, an owner and a test family; runs of that bug emit the signature
mixed with background noise, sometimes miss a line, and 'sibling' bugs share most of their signature, which is the
hard case in the real system. Not-a-bug runs are infrastructure or test-content problems."""
import random
from datetime import datetime, timedelta

DOMAINS = ["power", "memory", "reset", "virtualization", "media", "compute", "fabric", "firmware"]
COMPONENTS = ["gt", "dma", "pcie", "fw", "mem", "sched", "link", "irq", "pmu", "vf"]
SYMPTOMS = ["engine reset", "timeout waiting for fence", "page fault at <HEX>", "device lost", "link training failed",
            "ECC uncorrectable", "watchdog expired", "CRC mismatch", "thermal trip", "context banned"]
NOISE = ["test step <N> started", "allocating <N> buffers", "INFO: iteration <N> passed", "loading module", "heartbeat ok",
         "perf counter sample <N>", "config applied", "mounting share", "INFO: retrying connection"]
GENERIC = ["FAIL: test step <N> returned non-zero", "ERROR: timeout waiting for test harness", "FAIL: assertion in test script"]
INFRA = ["ERROR: node unreachable", "ERROR: container image pull failed", "FAIL: missing test binary", "ERROR: disk quota exceeded"]


def make_world(n_bugs=40, n_runs=1200, p_none=.18, seed=0):
    rng = random.Random(seed)
    t0 = datetime(2026, 1, 1)
    bugs = []
    for i in range(n_bugs):
        sib = bugs[rng.randrange(len(bugs))] if bugs and rng.random() < .35 else None
        comp = sib["component"] if sib else rng.choice(COMPONENTS)
        sig = list(sib["signature"][:2]) if sib else [f"{comp}: {rng.choice(SYMPTOMS)}", f"{comp}: {rng.choice(SYMPTOMS)} on <PCI>"]
        sig.append(f"{comp}: first fault code {rng.randrange(16, 255):#x}")
        bugs.append({"id": f"BUG-{i:03d}", "domain": sib["domain"] if sib else rng.choice(DOMAINS), "component": comp,
                     "owner": f"owner{rng.randrange(8)}", "family": f"T{rng.randrange(12)}", "signature": sig,
                     "created": t0 + timedelta(days=rng.randrange(0, 200)),
                     "title": f"[{comp}] {sig[0].split(': ', 1)[1]}", "root_cause": f"{comp} {sig[-1].split(': ', 1)[1]}"})
    w = [1 / (i + 1) ** .8 for i in range(n_bugs)]          # long tail: a few bugs own most failures
    runs = []
    for j in range(n_runs):
        start = t0 + timedelta(days=rng.uniform(0, 260))
        if rng.random() < p_none:
            lines, label, b = ([rng.choice(INFRA)] if rng.random() < .6 else [rng.choice(GENERIC)]), "NONE", None
        else:
            b = rng.choices(bugs, w)[0]
            if b["created"] > start + timedelta(days=30):
                start = b["created"] + timedelta(days=rng.uniform(0, 30))
            lines, label = [s for s in b["signature"] if rng.random() > .3] or b["signature"][:1], b["id"]
            if rng.random() < .25:
                lines.append(rng.choice(GENERIC + INFRA))      # product bugs also trip generic/infra-looking lines
        lines += rng.sample(NOISE, 4)
        rng.shuffle(lines)
        runs.append({"run": f"RUN-{j:05d}", "label": label, "start": start, "lines": lines,
                     "domain": b["domain"] if b else rng.choice(DOMAINS), "owner": b["owner"] if b else f"owner{rng.randrange(8)}",
                     "family": b["family"] if b and rng.random() < .7 else f"T{rng.randrange(12)}"})
    return bugs, runs


def split(runs, bugs, seed=0):
    """Grouped by bug: 30% of bugs are test-only, so the test split measures bugs never seen in training."""
    rng = random.Random(seed)
    ids = [b["id"] for b in bugs]
    rng.shuffle(ids)
    test_bugs = set(ids[: int(.3 * len(ids))])
    for r in runs:
        r["split"] = "test" if (r["label"] in test_bugs or (r["label"] == "NONE" and rng.random() < .3)) else "train"
        # frequent bugs also appear in test (seen-bug slice)
        if r["label"] not in test_bugs and r["label"] != "NONE" and rng.random() < .2:
            r["split"] = "test"
    return runs

In [ ]:
%%writefile ranker.py
"""Failure ranker (no LLM): failed run -> ranked list of known bugs + calibrated LINK / REVIEW / NEW decision.

Features per (run, candidate bug) pair:
  - log channel   : signature lines of the bug found in the run's normalised evidence (exact skeleton hits,
                    fuzzy phrase hits, idf-weighted), first-fault agreement
  - metadata prior: same test family, same domain, same owner, peer votes (how many earlier runs of this test
                    family were linked to the bug), recency of the bug's last link, bug created before the run
  - hard rules    : a bug tagged for another platform vendor never outranks any other candidate
Model: logistic regression over pair features (pairwise -> pointwise scores), grouped CV by bug, then a
calibrated open-set threshold: LINK only when P(top-1) >= tau, tau picked for >= 85 % out-of-fold precision.
"""
from __future__ import annotations

import math
import re
from collections import Counter, defaultdict

import numpy as np
from sklearn.linear_model import LogisticRegression

FEATS = ["sig_hits", "sig_frac", "sig_idf", "first_fault", "same_family", "same_domain", "same_owner",
         "peer_votes", "log_recency", "created_before"]


def tokens(s: str) -> set[str]:
    return set(re.findall(r"[a-z_]{3,}", s.lower()))


class Ranker:
    def __init__(self, C: float = 1.0, pool: int = 50):
        self.C, self.pool = C, pool

    # ------------------------------------------------------------- fit
    def fit(self, runs: list[dict], bugs: list[dict]):
        self.bugs = {b["id"]: b for b in bugs}
        self.df = Counter(t for b in bugs for s in b["signature"] for t in tokens(s))
        self.history = sorted((r for r in runs if r["label"] != "NONE"), key=lambda r: r["start"])
        X, y = [], []
        for r in runs:
            if r["label"] == "NONE":
                continue
            for bid in self._pool(r):
                X.append(self._feat(r, self.bugs[bid]))
                y.append(int(bid == r["label"]))
        self.clf = LogisticRegression(C=self.C, max_iter=2000, class_weight="balanced").fit(np.array(X), np.array(y))
        return self

    # ------------------------------------------------------------- features
    def _pool(self, r):
        return [b for b, v in self.bugs.items() if v["created"] <= r["start"]][: self.pool * 10]

    def _feat(self, r, b):
        text = " ".join(r["lines"]).lower()
        hits = [s for s in b["signature"] if s.lower() in text]
        idf = sum(math.log(1 + len(self.bugs) / (1 + self.df[t])) for s in hits for t in tokens(s))
        prev = [h for h in self.history if h["start"] < r["start"] and h["label"] == b["id"]]
        peers = sum(1 for h in prev if h["family"] == r["family"])
        last = (r["start"] - prev[-1]["start"]).days if prev else 999
        ff = next((l for l in r["lines"] if re.search(r"fault|reset|lost|timeout|mismatch", l)), "")
        return [len(hits), len(hits) / len(b["signature"]), idf, int(bool(ff) and ff in b["signature"]),
                int(b["family"] == r["family"]), int(b["domain"] == r["domain"]), int(b["owner"] == r["owner"]),
                math.log1p(peers), -math.log1p(last), int(b["created"] <= r["start"])]

    # ------------------------------------------------------------- predict
    def rank(self, r, k: int = 5):
        pool = self._pool(r)
        if not pool:
            return []
        p = self.clf.predict_proba(np.array([self._feat(r, self.bugs[b]) for b in pool]))[:, 1]
        order = np.argsort(-p)[:k]
        return [(pool[i], float(p[i])) for i in order]


def calibrate_tau(scored: list[tuple[float, bool]], target_precision: float = .85) -> float:
    """Lowest threshold whose out-of-fold precision on LINK decisions is >= target."""
    for tau in np.arange(.05, 1.0, .01):
        sel = [ok for s, ok in scored if s >= tau]
        if sel and sum(sel) / len(sel) >= target_precision:
            return float(tau)
    return 1.0


def evaluate(ranker: Ranker, runs: list[dict], unseen: set[str] | None = None) -> dict:
    res = defaultdict(list)
    for r in runs:
        if r["label"] == "NONE":
            continue
        top = [b for b, _ in ranker.rank(r, 3)]
        res["top1"].append(bool(top) and top[0] == r["label"])
        res["top3"].append(r["label"] in top)
        if unseen is not None and r["label"] in unseen:
            res["unseen_top1"].append(bool(top) and top[0] == r["label"])
    return {k: round(100 * sum(v) / len(v), 1) for k, v in res.items() if v}

In [ ]:
%%writefile matcher.py
"""Bug matcher: (root-cause analysis of a failed run, top-5 candidate bugs) -> {"bug": id | "NONE", "confidence", "why"}.

Qwen3-8B + QLoRA (Unsloth). The champion was reached in four data iterations, not by changing the model:
  v1  rejection sampling: keep only teacher trajectories whose final pick was right
  v2  cleaned data: STaR rationales (writer sees the gold, explains it from evidence; leaky or self-contradicting
      rationales filtered), one consistent policy for runs without logs, counterfactual augmentation
      (candidate order shuffled; truth removed -> target NONE), grouped validation split
  v3  2x data + out-of-fold ranker candidates (the matcher learns from the ranker's real mistakes) + a new
      platform slice; local 8B rationale writer instead of an API model (8B and 14B writers tied)
Serving: the matcher may override the ranker's top-1 only through a confidence gate tuned on validation.
In the full system it is the last model of the chain: it reads the DPO-tuned writer's root cause, the ranker's out-of-fold top-5, an
evidence probe (how many of each bug's known log lines appear in this run) and the deterministic graph facts of each candidate.
"""
from __future__ import annotations

import json
import os
import random
import re

PROMPT = """You match a finished root-cause analysis of ONE failed test run to the known bugs below.
Pick the bug that describes the SAME failure: same failing component, same origin and mechanism (what failed first and why).
A shared final symptom (timeout, hang, test FAIL, device lost, reset) is NOT enough: the origin must agree.
If no candidate shares the origin, answer NONE.

Root-cause analysis:
{rca}

Candidates (ranker order, ranker score, evidence probe, graph facts):
{cands}

Answer JSON only: {{"bug": "<id or NONE>", "confidence": 0-1, "why": "<one paragraph citing the deciding evidence>"}}"""


def render_candidates(cards: list[dict], probe: dict[str, str]) -> str:
    """One line per candidate; `facts` is the KG node's output, e.g. "same test: yes, same owner: no, filed before run: yes"."""
    return "\n".join(f"- {c['id']} [{c['score']:.2f}] {c['title']} | first fault: {c.get('first_fault', '?')} | "
                     f"signature lines found in this run: {probe.get(c['id'], 'unknown')} | graph facts: {c.get('facts', 'none')}"
                     for c in cards)


# ------------------------------------------------------------------ dataset recipe
def counterfactuals(row: dict, rng: random.Random) -> list[dict]:
    """Two cheap augmentations that taught the model the NONE answer and removed position bias."""
    out = []
    c = row["cands"][:]
    rng.shuffle(c)
    out.append({**row, "cands": c})
    if row["gold"] != "NONE":
        c2 = [x for x in row["cands"] if x["id"] != row["gold"]]
        out.append({**row, "cands": c2, "gold": "NONE",
                    "target": json.dumps({"bug": "NONE", "confidence": .3, "why": "No candidate shares the origin found in the analysis."})})
    return out


def star_rationale(row: dict, writer) -> str | None:
    """STaR-style rationale: the writer sees the gold answer and must justify it from the evidence only.
    Rejected when it leaks the answer without evidence, contradicts itself, or is boilerplate."""
    why = writer(f"Explain from the evidence why {row['gold']} is the right bug.\n{row['rca']}")
    if row["gold"] not in why or re.search(r"(?i)as (stated|given)|the answer is|cannot determine", why):
        return None
    return why


def to_chat(row: dict) -> dict:
    return {"messages": [{"role": "user", "content": PROMPT.format(rca=row["rca"], cands=render_candidates(row["cands"], row.get("probe", {})))},
                         {"role": "assistant", "content": row["target"]}]}


# ------------------------------------------------------------------ training (Unsloth QLoRA)
def train(rows: list[dict], out: str, base: str = "Qwen/Qwen3-8B", maxlen: int = 8192, epochs: int = 3, lr: float = 1e-4):
    import torch
    from datasets import Dataset
    from trl import SFTConfig, SFTTrainer
    from unsloth import FastLanguageModel

    model, tok = FastLanguageModel.from_pretrained(base, max_seq_length=maxlen, load_in_4bit=True, dtype=torch.float16)
    model = FastLanguageModel.get_peft_model(model, r=16, lora_alpha=32, lora_dropout=0.0,
                                             target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
                                             use_gradient_checkpointing="unsloth", random_state=0)
    def fmt(r):  # prompt/completion rows: the loss is on the answer tokens only
        m = to_chat(r)["messages"]
        return {"prompt": tok.apply_chat_template(m[:1], tokenize=False, add_generation_prompt=True, enable_thinking=False),
                "completion": m[1]["content"] + tok.eos_token}
    tr = Dataset.from_list([fmt(r) for r in rows if r["split"] == "train"])
    va = Dataset.from_list([fmt(r) for r in rows if r["split"] == "val"])
    cfg = SFTConfig(output_dir=out, per_device_train_batch_size=1, gradient_accumulation_steps=8, num_train_epochs=epochs,
                    learning_rate=lr, lr_scheduler_type="cosine", warmup_ratio=.05, logging_steps=5,
                    eval_strategy="epoch", save_strategy="epoch", fp16=True, max_length=maxlen,
                    completion_only_loss=True, report_to=[])
    SFTTrainer(model=model, processing_class=tok, train_dataset=tr, eval_dataset=va, args=cfg).train()
    model.save_pretrained(f"{out}/adapter")


# ------------------------------------------------------------------ serving
def parse(text: str) -> dict:
    m = re.search(r"\{.*\}", text, re.S)
    try:
        return json.loads(m.group(0)) if m else {}
    except json.JSONDecodeError:
        return {}


def gated_pick(ranker_top: list[tuple[str, float]], answer: dict, tau: float = .7) -> str:
    """Override gate: the ranker's top-1 stands unless the matcher names another candidate with confidence >= tau.
    A NONE answer is surfaced as a note but never removes a confident ranker link."""
    pick, conf = answer.get("bug"), float(answer.get("confidence") or 0)
    ids = [b for b, _ in ranker_top]
    if pick in ids and conf >= tau:
        return pick
    return ids[0] if ids else "NONE"

## 1. Dataset: ranker candidates → prompts, with counterfactual augmentation

In [ ]:
import json, random
from synthetic import make_world, split
from ranker import Ranker
from matcher import counterfactuals, PROMPT, render_candidates, train, parse, gated_pick
bugs, runs = make_world(n_bugs=40, n_runs=900, seed=4); runs = split(runs, bugs)
B = {b["id"]: b for b in bugs}
rk = Ranker().fit([r for r in runs if r["split"] == "train"], bugs)
rng = random.Random(0); rows = []
for r in runs:
    top = rk.rank(r, 5)
    if not top: continue
    cands = [{**B[b], "score": s, "first_fault": B[b]["signature"][-1],
              "facts": f"same test family: {'yes' if B[b]['family'] == r['family'] else 'no'}, same owner: {'yes' if B[b]['owner'] == r['owner'] else 'no'}"}
             for b, s in top]
    gold = r["label"] if r["label"] in [c["id"] for c in cands] else "NONE"
    rca = "Evidence: " + " | ".join(l for l in r["lines"] if ":" in l)     # stand-in for the RCA writer's output
    row = {"split": r["split"], "rca": rca, "cands": cands, "gold": gold, "label": r["label"], "top": top,
           "target": json.dumps({"bug": gold, "confidence": .9 if gold != "NONE" else .3, "why": "origin matches" if gold != "NONE" else "no shared origin"})}
    rows += [row] + (counterfactuals(row, rng) if r["split"] == "train" else [])
for r in rows:
    if r["split"] == "test" and rng.random() < .3: r["split"] = "val"
print(len(rows), sum(r["split"] == "train" for r in rows), "train rows")

## 2. Train (≈10 min on a T4 with 0.6B)

In [ ]:
BASE = "unsloth/Qwen3-0.6B"   # production: Qwen/Qwen3-8B on a 24 GB GPU
train(rows, "matcher-out", base=BASE, maxlen=2048, epochs=2, lr=2e-4)

## 3. Evaluate: matcher alone vs matcher behind the confidence gate

In [ ]:
import torch
from unsloth import FastLanguageModel
model, tok = FastLanguageModel.from_pretrained("matcher-out/adapter", max_seq_length=2048, load_in_4bit=True)
FastLanguageModel.for_inference(model)
def ask(row):
    p = PROMPT.format(rca=row["rca"], cands=render_candidates(row["cands"], {}))   # same prompt as training, probe unknown here
    x = tok.apply_chat_template([{"role": "user", "content": p}], add_generation_prompt=True, return_tensors="pt", enable_thinking=False).to("cuda")
    y = model.generate(x, max_new_tokens=120, do_sample=False)
    return parse(tok.decode(y[0, x.shape[1]:], skip_special_tokens=True))
test = [r for r in rows if r["split"] == "test" and r["label"] != "NONE"]
ans = [ask(r) for r in test]
ranker = sum(r["top"][0][0] == r["label"] for r in test) / len(test)
alone = sum(a.get("bug") == r["label"] for a, r in zip(ans, test)) / len(test)
gate = sum(gated_pick(r["top"], a) == r["label"] for a, r in zip(ans, test)) / len(test)
print(f"ranker {ranker:.1%} | matcher {alone:.1%} | matcher + gate {gate:.1%}")